# Mutual Fund Analytics - Getting Started

Welcome to the **Mutual Fund Analytics Platform**! This notebook acts as your exploratory playground. It demonstrates how to import and use the python files we set up in `src/`, generate some mock mutual fund performance data, calculate analytics metrics, and plot the comparative charts.

### Step 1: Environment Setup
First, we will add the project root directory to Python's system path so that we can easily load modules from our custom `src/` folder. Then, we import all required libraries.

In [ ]:
import sys
from pathlib import Path

# Find the folder where this notebook is, and get its parent folder (the project root)
notebook_dir = Path.cwd()
project_root = notebook_dir.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import our custom helper functions and analytical formulas
from src.utils import get_data_dir, load_csv
from src.analytics import (
    calculate_cagr,
    calculate_annualized_return,
    calculate_annualized_volatility,
    calculate_sharpe_ratio,
    calculate_beta,
    calculate_alpha
)

# Set a premium visual style for our charts
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("Setup completed successfully!")

### Step 2: Create a Mock Mutual Fund Dataset
Since we don't have real mutual fund CSVs in our data folders yet, let's programmatically generate a realistic daily returns dataset for a Mutual Fund and its Benchmark Index (e.g. Nifty 50) and save it to the `data/raw/` folder.

In [ ]:
# Set random seed so the numbers are the same every time we run the cell
np.random.seed(42)

# Generate dates for 252 business days (about 1 standard trading year)
dates = pd.date_range(start="2025-01-01", periods=252, freq="B")

# Simulate market index daily returns (annual return ~12%, annualized risk/volatility ~15%)
market_mean = 0.12 / 252
market_vol = 0.15 / np.sqrt(252)
market_returns = np.random.normal(loc=market_mean, scale=market_vol, size=252)

# Simulate mutual fund daily returns 
# (with a beta of 1.1, meaning it is 1.1x as sensitive to the market, and 3% annualized excess returns/alpha)
fund_returns = 1.1 * market_returns + np.random.normal(loc=0.03/252, scale=0.05/np.sqrt(252), size=252)

# Assemble into a table
df = pd.DataFrame({
    'Date': dates,
    'Fund_Returns': fund_returns,
    'Market_Returns': market_returns
})

# Save it to 'data/raw/mock_returns.csv' using our dynamic path utility
raw_data_dir = get_data_dir("raw")
file_path = raw_data_dir / "mock_returns.csv"
df.to_csv(file_path, index=False)
print(f"Mock returns CSV successfully saved at: {file_path}")

### Step 3: Load Data and Compute Cumulative Performance
We use our custom `load_csv` helper function to read our data back in, convert date formats, and calculate cumulative growth over time.

In [ ]:
# Load data using our utility
data = load_csv("mock_returns.csv", "raw")

# Ensure Date is read as a date object
data['Date'] = pd.to_datetime(data['Date'])

# Calculate cumulative returns (the growth of $1 over time)
data['Fund_Cumulative'] = (1 + data['Fund_Returns']).cumprod() - 1
data['Market_Cumulative'] = (1 + data['Market_Returns']).cumprod() - 1

data.head()

### Step 4: Calculate Mutual Fund Risk & Return Metrics
Let's pass our returns data into the analytical functions in `src/analytics.py` to evaluate the fund's performance.

In [ ]:
# Number of business/trading periods per year
trading_days = 252

# Calculate values
fund_ann_return = calculate_annualized_return(data['Fund_Returns'], trading_days)
market_ann_return = calculate_annualized_return(data['Market_Returns'], trading_days)

fund_vol = calculate_annualized_volatility(data['Fund_Returns'], trading_days)
market_vol = calculate_annualized_volatility(data['Market_Returns'], trading_days)

sharpe = calculate_sharpe_ratio(data['Fund_Returns'], risk_free_rate=0.05, periods_per_year=trading_days)
beta = calculate_beta(data['Fund_Returns'], data['Market_Returns'])
alpha = calculate_alpha(data['Fund_Returns'], data['Market_Returns'], risk_free_rate=0.05, periods_per_year=trading_days)

print("==================================================")
print("           MUTUAL FUND ANALYTICS METRICS          ")
print("==================================================")
print(f"Annualized Fund Return:    {fund_ann_return:.2%}")
print(f"Annualized Market Return:  {market_ann_return:.2%}")
print(f"Annualized Volatility:     {fund_vol:.2%} (Market Volatility: {market_vol:.2%})")
print(f"Sharpe Ratio (Rf = 5%):    {sharpe:.2f} (Higher means better risk-adjusted return)")
print(f"Beta (Sensitivity):        {beta:.2f} (Above 1.0 means more volatile than market)")
print(f"Jensen's Alpha:            {alpha:.2%} (Positive means outperforming expectations)")
print("==================================================")

### Step 5: Visualize Performance Comparison
Finally, let's create a premium quality comparative chart showing how our mutual fund performed against the market benchmark over the course of the year, and save the image to the `docs/` folder.

In [ ]:
plt.figure(figsize=(12, 6))

# Plot the mutual fund in premium blue and the benchmark in slate gray
plt.plot(data['Date'], data['Fund_Cumulative'] * 100, label=f"Mutual Fund (Return: {fund_ann_return:.1%})", color="#1a73e8", linewidth=2.5)
plt.plot(data['Date'], data['Market_Cumulative'] * 100, label=f"Market Benchmark (Return: {market_ann_return:.1%})", color="#5f6368", linestyle="--", linewidth=1.5)

# Formatting details
plt.title("Mutual Fund Cumulative Performance vs. Market Index", fontsize=15, fontweight="bold", pad=15)
plt.xlabel("Timeline (Date)", fontsize=12)
plt.ylabel("Cumulative Returns (% Growth)", fontsize=12)
plt.legend(loc="upper left", fontsize=11, frameon=True)
plt.grid(True, linestyle=":", alpha=0.6)

# Ensure layout fits and save image
plt.tight_layout()
docs_dir = project_root / "docs"
docs_dir.mkdir(exist_ok=True)
plt.savefig(docs_dir / "fund_vs_benchmark.png", dpi=300)
plt.show()